# Profit Desk validation

## tl;dr

The launch decision is **shadow-only, 0u**. This notebook profiles the committed outcome ledger and makes the calibration exclusion correction inspectable. It does not claim that a historical positive return proves future profit.

## Context & Methods

Decision: whether current evidence is strong enough to publish live profit-qualified picks. The controlling source is `data/calibration/outcome_ledger.json`; price and probability provenance are recovered from each immutable `pregame_snapshot` when present.

### Key Assumptions

- A settled row is `win` or `loss`.
- A row is only eligible for the shared calibrator when `calibration_eligible is True`.
- Probabilities owned by `player_props_ml_v1` are evaluated separately and never train the shared Platt layer.
- A posted two-sided market requires both offered sides; a one-sided implied probability is not called no-vig.

## Data

In [1]:
from pathlib import Path
import json

repo_root = Path.cwd()
ledger_path = repo_root / 'data' / 'calibration' / 'outcome_ledger.json'
ledger = json.loads(ledger_path.read_text(encoding='utf-8'))
records = [row for row in ledger.get('records', []) if isinstance(row, dict)]
settled = [row for row in records if row.get('result') in {'win', 'loss'}]
ledger.get('summary', {}), len(records), len(settled)

({'decided_picks': 2406,
  'pending_picks': 2519,
  'total_picks': 5005,
  'trainable_decided_picks': 1404},
 5005,
 2406)

## Results

In [2]:
def snapshot(row):
    value = row.get('pregame_snapshot')
    return value if isinstance(value, dict) else {}

def first(row, *keys):
    snap = snapshot(row)
    for key in keys:
        value = row.get(key)
        if value not in (None, ''):
            return value
        value = snap.get(key)
        if value not in (None, ''):
            return value
    return None

def is_ml_owned(row):
    return (
        first(row, 'ml_calibration_excluded') is True
        or str(first(row, 'probability_source') or '') == 'player_props_ml_v1'
    )

def has_paired_price(row):
    over = first(row, 'market_over_odds', 'over_odds')
    under = first(row, 'market_under_odds', 'under_odds')
    selected = first(row, 'selected_odds', 'market_selected_odds')
    opposite = first(row, 'opposite_odds', 'market_opposite_odds')
    return (over not in (None, '') and under not in (None, '')) or (selected not in (None, '') and opposite not in (None, ''))

old_trainable = [
    row for row in settled
    if isinstance(row.get('raw_probability'), (int, float))
    and row.get('calibration_eligible') is not False
]
corrected_trainable = [
    row for row in settled
    if isinstance(row.get('raw_probability'), (int, float))
    and row.get('calibration_eligible') is True
    and not is_ml_owned(row)
]
profile = {
    'ledger_total': len(records),
    'settled_win_loss': len(settled),
    'rows_with_odds': sum(first(row, 'odds') not in (None, '', 0) for row in settled),
    'paired_price_rows': sum(has_paired_price(row) for row in settled),
    'ml_owned_rows': sum(is_ml_owned(row) for row in settled),
    'legacy_trainable_rule_rows': len(old_trainable),
    'strict_explicitly_eligible_rows_in_persisted_ledger': len(corrected_trainable),
}
print(json.dumps(profile, indent=2, sort_keys=True))

{
  "ledger_total": 5005,
  "legacy_trainable_rule_rows": 1404,
  "ml_owned_rows": 618,
  "paired_price_rows": 364,
  "rows_with_odds": 2297,
  "settled_win_loss": 2406,
  "strict_explicitly_eligible_rows_in_persisted_ledger": 0
}


In [3]:
assert profile['ledger_total'] == ledger['summary']['total_picks']
assert profile['settled_win_loss'] == ledger['summary']['decided_picks']
assert profile['legacy_trainable_rule_rows'] == ledger['summary']['trainable_decided_picks']
assert profile['strict_explicitly_eligible_rows_in_persisted_ledger'] <= profile['legacy_trainable_rule_rows']
print('Reasonableness checks passed.')

Reasonableness checks passed.


## Takeaways

- The persisted ledger predates the stricter explicit-eligibility contract; the normal calibration workflow must rebuild it before the corrected training count is authoritative.
- Price coverage alone does not establish a fair-market baseline. Profit Desk separately requires a fresh paired/no-vig price and strictly prior compatible evidence.
- The launch remains shadow-only: no live-qualified picks and no nonzero stakes. Promotion requires prospectively stable evidence under the frozen policy version.